#### Imports


In [ ]:
from pathlib import Path
import json
import numpy as np
import tensorflow as tf
import pandas as pd
import random
import matplotlib.pyplot as plt
import datetime
import io
from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
%load_ext tensorboard

# Full data loader

In [ ]:
dataset_dirs = {
    "phone": Path("har_stage2_production_phones_v1"),
    "watch": Path("har_stage2_production_watch_v1"),
}

class_names = [
    "bike",
    "sit",
    "stairsdown",
    "stairsup",
    "stand",
    "walk"
]

label_to_index = {
    "bike": 0,
    "sit": 1,
    "stairsdown": 2,
    "stairsup": 3,
    "stand": 4,
    "walk": 5
}

all_X = []
all_y = []
all_participants = []
all_sources = []
all_metadata = []


for source_name, dataset_dir in dataset_dirs.items():
    print(f"\nLoading {source_name}: {dataset_dir}")

    shard_paths = sorted(dataset_dir.glob("windows-*.npz"))
    print("Shards found:", len(shard_paths))

    source_X = []
    source_y = []
    source_participants = []

    for shard_path in shard_paths:
        with np.load(shard_path, allow_pickle=False) as z:
            X = z["X"].astype(np.float32)
            labels = z["labels"]
            metadata = z["metadata"]

            y = np.array([
                label_to_index[str(label)]
                for label in labels
            ], dtype=np.int64)

            participants = np.array([
                json.loads(item)["User"]
                for item in metadata
            ])
            
            all_metadata.extend(metadata)

        source_X.append(X)
        source_y.append(y)
        source_participants.append(participants)

    X_source = np.concatenate(source_X, axis=0)
    y_source = np.concatenate(source_y, axis=0)
    participants_source = np.concatenate(source_participants, axis=0)

    print("Windows:", X_source.shape[0])
    print("X shape:", X_source.shape)
    print("Participants:", np.unique(participants_source))

    all_X.append(X_source)
    all_y.append(y_source)
    all_participants.append(participants_source)
    all_sources.append(
        np.full(len(X_source), source_name, dtype="<U16")
    )

X = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)
participant_ids = np.concatenate(all_participants, axis=0)
source_ids = np.concatenate(all_sources, axis=0)
metadata = np.array(all_metadata)

print("\nCombined dataset")
print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("Participants:", np.unique(participant_ids))
print("Sources:", np.unique(source_ids))
print("Windows per source:")

print("Metadata records:", len(metadata))
print("Windows:", len(X))

assert len(metadata) == len(X)

for source in np.unique(source_ids):
    print(f"{source}: {np.sum(source_ids == source)}")

#### To see specific devices

In [ ]:
device_ids = np.array([
    json.loads(item)["Device"]
    for item in metadata
])

print("device_ids shape:", device_ids.shape)
print("Unique devices:", np.unique(device_ids))

assert device_ids.shape[0] == X.shape[0]

In [ ]:
assert X.ndim == 3
assert X.shape[1:] == (128, 6)
assert X.dtype == np.float32

assert y.ndim == 1
assert len(y) == len(X)
assert len(participant_ids) == len(X)
assert len(source_ids) == len(X)

assert set(np.unique(y)) == set(range(6))
assert set(np.unique(source_ids)) == {"phone", "watch"}

In [ ]:
phone_count = int(np.sum(source_ids == "phone"))
watch_count = int(np.sum(source_ids == "watch"))

print(f"Phone windows: {phone_count}")
print(f"Watch windows: {watch_count}")
print(f"Total windows: {len(X)}")

assert phone_count == 32160, (
    f"Expected 32160 phone windows, got {phone_count}"
)

assert watch_count == 5032, (
    f"Expected 5032 watch windows, got {watch_count}"
)

assert len(X) == 37192, (
    f"Expected 37192 total windows, got {len(X)}"
)

print("Participants:", np.unique(participant_ids))
print("Sources:", np.unique(source_ids))
print("Phone windows:", phone_count)
print("Watch windows:", watch_count)

In [ ]:
print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("Participants:", np.unique(participant_ids))
print("Sources:", np.unique(source_ids))

print("\nWindows per source:")
for source in np.unique(source_ids):
    print(source, np.sum(source_ids == source))

print("\nWindows per class:")
for index, name in enumerate(class_names):
    print(name, np.sum(y == index))

# Split data


In [ ]:
import numpy as np
import pandas as pd

# Put participants in a stable order
unique_participants = np.unique(participant_ids)

# Use the next participant as validation:
# test a -> validate b, test b -> validate c, etc.
validation_participant_for = {
    participant: unique_participants[(index + 1) % len(unique_participants)]
    for index, participant in enumerate(unique_participants)
}

fold_results = []

for test_participant in unique_participants:
    validation_participant = validation_participant_for[test_participant]

    test_mask = participant_ids == test_participant
    validation_mask = participant_ids == validation_participant
    train_mask = (
        (participant_ids != test_participant)
        & (participant_ids != validation_participant)
    )

    train_idx = np.where(train_mask)[0]
    val_idx = np.where(validation_mask)[0]
    test_idx = np.where(test_mask)[0]

    print(
        f"\nFold: test={test_participant}, "
        f"validation={validation_participant}"
    )
    print("Train participants:", np.unique(participant_ids[train_idx]))
    print("Train windows:", len(train_idx))
    print("Validation windows:", len(val_idx))
    print("Test windows:", len(test_idx))

    # Verify participant isolation
    assert test_participant not in participant_ids[train_idx]
    assert validation_participant not in participant_ids[train_idx]
    assert test_participant not in participant_ids[val_idx]
    assert validation_participant not in participant_ids[test_idx]

    # Store the indices for this fold
    fold_results.append({
        "test_participant": test_participant,
        "validation_participant": validation_participant,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
    })

#### Verify


In [ ]:
for fold in fold_results:
    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]
    test_idx = fold["test_idx"]

    train_participants = set(np.unique(participant_ids[train_idx]))
    val_participants = set(np.unique(participant_ids[val_idx]))
    test_participants = set(np.unique(participant_ids[test_idx]))

    print(
        f"\nFold: test={fold['test_participant']}, "
        f"validation={fold['validation_participant']}"
    )

    print("Train participants:", sorted(train_participants))
    print("Validation participants:", sorted(val_participants))
    print("Test participants:", sorted(test_participants))

    print("\nTrain windows:", len(train_idx))
    print("Validation windows:", len(val_idx))
    print("Test windows:", len(test_idx))

    assert train_participants.isdisjoint(val_participants)
    assert train_participants.isdisjoint(test_participants)
    assert val_participants.isdisjoint(test_participants)

In [ ]:
for fold in fold_results:
    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]
    test_idx = fold["test_idx"]

    print(
        f"\n{'=' * 70}\n"
        f"Fold: test={fold['test_participant']}, "
        f"validation={fold['validation_participant']}\n"
        f"{'=' * 70}"
    )

    for split_name, split_idx in [
        ("Train", train_idx),
        ("Validation", val_idx),
        ("Test", test_idx),
    ]:
        y_split = y[split_idx]

        print(f"\n{split_name} class counts:")

        for index, name in enumerate(class_names):
            print(f"{name}: {np.sum(y_split == index)}")

# Documentation callback class 

In [ ]:
class HARDiagnosticCallback(tf.keras.callbacks.Callback):
    def __init__(self, log_dir, X_sample, y_sample, X_test, y_test, source_test, class_names):
        super().__init__()
        self.writer = tf.summary.create_file_writer(log_dir)
        # Small fixed batch to compute exact gradients at the end of each epoch
        self.X_sample = tf.convert_to_tensor(X_sample[:256])
        self.y_sample = tf.convert_to_tensor(y_sample[:256])
        self.X_test = X_test
        self.y_test = y_test
        self.source_test = source_test
        self.class_names = class_names
        self.prev_weights = None

    def on_epoch_begin(self, epoch, logs=None):
        self.prev_weights = [tf.identity(w) for w in self.model.trainable_weights]

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        
        # 1. Compute layer-wise gradients on a fixed diagnostic batch
        with tf.GradientTape() as tape:
            preds = self.model(self.X_sample, training=True)
            loss = self.model.compiled_loss(self.y_sample, preds)
        grads = tape.gradient(loss, self.model.trainable_weights)

        with self.writer.as_default():
            # 2. Log current Learning Rate (crucial for spotting error valley dampening)
            lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
            tf.summary.scalar("optimization/learning_rate", lr, step=epoch)

            # 3. Log Gradient L2 Norms & Relative Weight Updates per layer
            global_grad_norm = tf.linalg.global_norm([g for g in grads if g is not None])
            tf.summary.scalar("gradients/global_l2_norm", global_grad_norm, step=epoch)

            for weight, grad, prev_w in zip(self.model.trainable_weights, grads, self.prev_weights):
                if grad is None:
                    continue
                layer_name = weight.name.replace(":", "_")
                grad_norm = tf.linalg.global_norm([grad])
                weight_norm = tf.linalg.global_norm([weight])
                update_norm = tf.linalg.global_norm([weight - prev_w])
                
                # Healthy ratio is typically ~1e-3
                update_ratio = update_norm / (weight_norm + 1e-8)

                tf.summary.scalar(f"gradients_norm/{layer_name}", grad_norm, step=epoch)
                tf.summary.scalar(f"update_ratio/{layer_name}", update_ratio, step=epoch)
                tf.summary.histogram(f"gradients_hist/{layer_name}", grad, step=epoch)

            # 4. Log Generalization Gap (Val Loss - Train Loss)
            if "loss" in logs and "val_loss" in logs:
                tf.summary.scalar("generalization/loss_gap", logs["val_loss"] - logs["loss"], step=epoch)

        self.writer.flush()

    def on_train_end(self, logs=None):
        # 5. Final Held-Out Test Evaluation (Confusion Matrix + Phone vs. Watch Breakdown)
        y_pred = np.argmax(self.model.predict(self.X_test, verbose=0), axis=1)
        
        test_acc = accuracy_score(self.y_test, y_pred)
        test_f1 = f1_score(self.y_test, y_pred, average="macro", zero_division=0)

        # Device-specific accuracy (Phone vs Watch)
        phone_mask = self.source_test == "phone"
        watch_mask = self.source_test == "watch"
        phone_acc = accuracy_score(self.y_test[phone_mask], y_pred[phone_mask]) if np.any(phone_mask) else 0.0
        watch_acc = accuracy_score(self.y_test[watch_mask], y_pred[watch_mask]) if np.any(watch_mask) else 0.0

        # Render Confusion Matrix to PNG buffer for TensorBoard Images tab
        cm = confusion_matrix(self.y_test, y_pred, labels=np.arange(len(self.class_names)))
        fig, ax = plt.subplots(figsize=(6, 6))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks(np.arange(len(self.class_names)))
        ax.set_yticks(np.arange(len(self.class_names)))
        ax.set_xticklabels(self.class_names, rotation=45, ha="right")
        ax.set_yticklabels(self.class_names)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"Test Confusion Matrix (Macro F1: {test_f1:.3f})")

        for i in range(len(self.class_names)):
            for j in range(len(self.class_names)):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")

        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format="png")
        plt.close(fig)
        buf.seek(0)
        cm_image = tf.image.decode_png(buf.getvalue(), channels=4)
        cm_image = tf.expand_dims(cm_image, 0)

        with self.writer.as_default():
            tf.summary.scalar("test_final/accuracy", test_acc, step=0)
            tf.summary.scalar("test_final/macro_f1", test_f1, step=0)
            tf.summary.scalar("test_final/phone_accuracy", phone_acc, step=0)
            tf.summary.scalar("test_final/watch_accuracy", watch_acc, step=0)
            tf.summary.image("test_final/confusion_matrix", cm_image, step=0)

        self.writer.flush()
        self.writer.close()

# Normalization, CNN-LSTM and training loop
Combined for 9-fold leave-one-participant-out (LOPO)

In [ ]:
# Set seeds before each fold for more reproducible training
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Create the activity-name list in the same order as the model outputs
label_names = [None] * len(label_to_index)

for label, index in label_to_index.items():
    label_names[index] = label

NUM_CLASSES = len(label_names)
INPUT_SHAPE = (X.shape[1], X.shape[2])

# Store results from every leave-one-participant-out fold
lopo_results = []

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

OUTPUT_DIR = Path(rf"logs/run_{timestamp}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for fold in fold_results:
    test_participant = fold["test_participant"]
    validation_participant = fold["validation_participant"]

    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]
    test_idx = fold["test_idx"]

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_val = X[val_idx]
    y_val = y[val_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]

    print(
        f"\n{'=' * 70}\n"
        f"Fold: test={test_participant}, validation={validation_participant}\n"
        f"{'=' * 70}"
    )

    print("Train participants:", np.unique(participant_ids[train_idx]))
    print("Train windows:", len(train_idx))
    print("Validation windows:", len(val_idx))
    print("Test windows:", len(test_idx))

    # Fit normalization using only this fold's training participants
    normalizer = layers.Normalization(axis=-1)
    normalizer.adapt(X_train)

    # Create a completely fresh model for this fold
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),

        normalizer,

        layers.Conv1D(64, kernel_size=5, padding="same", activation="relu"),
        layers.BatchNormalization(),

        layers.Conv1D(128, kernel_size=5, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.SpatialDropout1D(0.4),

        layers.LSTM(64),

        layers.Dense(64, activation="relu"),
        layers.Dropout(0.4),

        layers.Dense(NUM_CLASSES, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )

    # Extract device sources for this fold's test participant
    source_test = source_ids[test_idx]
    run_timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = rf"{OUTPUT_DIR}/{run_timestamp}/fold_test_{test_participant}_val_{validation_participant}"

    tensorboard_callback = tf.keras.callbacks.TensorBoard(
        log_dir=log_dir,
        histogram_freq=1,
        write_graph=True,
        update_freq="epoch"
    )

    diagnostic_callback = HARDiagnosticCallback(
        log_dir=log_dir,
        X_sample=X_train,
        y_sample=y_train,
        X_test=X_test,
        y_test=y_test,
        source_test=source_test,
        class_names=label_names
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=64,
        callbacks=[early_stopping, reduce_lr, tensorboard_callback, diagnostic_callback],
        verbose=1
    )

    # Evaluate only on this fold's held-out test participant
    y_pred = np.argmax(
        model.predict(X_test, verbose=0),
        axis=1
    )

    fold_accuracy = accuracy_score(y_test, y_pred)
    fold_macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    print(
        f"\nTest participant {test_participant}: "
        f"accuracy={fold_accuracy:.4f}, "
        f"macro_f1={fold_macro_f1:.4f}"
    )

    print(
        "\nClassification report:\n",
        classification_report(
            y_test,
            y_pred,
            labels=np.arange(NUM_CLASSES),
            target_names=label_names,
            zero_division=0
        )
    )

    print(
        "Confusion matrix:\n",
        confusion_matrix(
            y_test,
            y_pred,
            labels=np.arange(NUM_CLASSES)
        )
    )

    # Save everything needed for later aggregation
    lopo_results.append({
        "test_participant": test_participant,
        "validation_participant": validation_participant,
        "test_idx": test_idx,
        "y_true": y_test,
        "y_pred": y_pred,
        "accuracy": fold_accuracy,
        "macro_f1": fold_macro_f1,
        "history": history.history
    })

# Aggregate results across all nine folds
accuracies = np.array([
    result["accuracy"]
    for result in lopo_results
])

macro_f1s = np.array([
    result["macro_f1"]
    for result in lopo_results
])

print("\n" + "=" * 70)
print("Leave-one-participant-out results")
print("=" * 70)

print(f"Mean test accuracy: {accuracies.mean():.4f} ± {accuracies.std():.4f}")
print(f"Mean test macro F1: {macro_f1s.mean():.4f} ± {macro_f1s.std():.4f}")

print("\nPer-participant results:")

for result in lopo_results:
    print(
        f"{result['test_participant']}: "
        f"accuracy={result['accuracy']:.4f}, "
        f"macro_f1={result['macro_f1']:.4f}"
    )

# Predictions and confusion matrix

In [ ]:
# Create a table with one row per leave-one-participant-out fold
lopo_table = pd.DataFrame([
    {
        "Test participant": result["test_participant"],
        "Validation participant": result["validation_participant"],
        "Test windows": len(result["test_idx"]),
        "Accuracy": result["accuracy"],
        "Macro F1": result["macro_f1"],
        "Epochs trained": len(result["history"]["loss"])
    }
    for result in lopo_results
])

# Sort by test participant for easier reading
lopo_table = lopo_table.sort_values("Test participant").reset_index(drop=True)

print("Leave-one-participant-out results:\n")
print(lopo_table.to_string(index=False))

print(
    f"\nMean accuracy: {lopo_table['Accuracy'].mean():.4f} "
    f"± {lopo_table['Accuracy'].std():.4f}"
)

print(
    f"Mean macro F1: {lopo_table['Macro F1'].mean():.4f} "
    f"± {lopo_table['Macro F1'].std():.4f}"
)

# Plot accuracy and macro F1 for every held-out test participant
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

bar_labels = [
    f"Test: {row['Test participant']}\nVal: {row['Validation participant']}"
    for _, row in lopo_table.iterrows()
]

axes[0].bar(
    bar_labels,
    lopo_table["Accuracy"],
    color="#4C72B0"
)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Test accuracy by held-out participant")
axes[0].tick_params(axis="x", rotation=45)

for index, value in enumerate(lopo_table["Accuracy"]):
    axes[0].text(
        index,
        value + 0.02,
        f"{value:.3f}",
        ha="center",
        fontsize=9
    )

axes[1].bar(
    bar_labels,
    lopo_table["Macro F1"],
    color="#DD8452"
)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Macro F1")
axes[1].set_title("Test macro F1 by held-out participant")
axes[1].tick_params(axis="x", rotation=45)

for index, value in enumerate(lopo_table["Macro F1"]):
    axes[1].text(
        index,
        value + 0.02,
        f"{value:.3f}",
        ha="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/fit